In [19]:
import pandas as pd
import numpy as np
import re
from pathlib import Path 


In [20]:
# Labels for answer choices the data leaves without one: a bare number ('02') or no value label at all (NaN).
# A scale is identified by its labelled choices, in code order; its entry gives the label of every unlabelled code.
# The same scale gets the same labels in any dataset. Labelled choices are never changed.
LABEL_BANK = {
    ("None", "1 child", "2 children"): {
        3: "3 children",
        4: "4 children",
        5: "5 children",
        6: "6 children",
        7: "7 children",
        8: "8 children",
        9: "9 children",
        10: "10 children",
    },
    ("Yes, they definitely should", "No, they definitely should not"): {
        2: "Yes, they should",
        3: "Yes, they probably should",
        4: "Undecided",
        5: "No, they probably should not",
        6: "No, they should not",
    },
    ("You cannot rely on either of them", "You can definitely rely more on close relatives",
     "You can rely equally on both", "You can definitely rely more on close friends"): {
        2: "You can rely more on close relatives",
        3: "You can probably rely more on close relatives",
        5: "You can probably rely more on close friends",
        6: "You can rely more on close friends",
    },
    ("Close relatives should definitely be the most important", "Both should be equally important",
     "Close friends should definitely be the most important"): {
        2: "Close relatives should be the most important",
        3: "Close relatives should probably be the most important",
        5: "Close friends should probably be the most important",
        6: "Close friends should be the most important",
    },
}

BRACKETS = {"[COUNTRY's]": "{country}'s", "[COUNTRY]": "{country}", "[CNTRY]": "{country}",
            "[EU/ USMCA/ etc.]": "{association}"}                       # the data's placeholders -> the bank's

def fill_placeholders(text, fields, where=""):
    """Text with this country's names in place of placeholders: the bank's {country}, {country}'s, {nationality},
    {parliament}, {association}, and the data's own bracket forms (BRACKETS). A 'the' that starts a sentence is
    capitalised. Prints every placeholder filled (where: e.g. the question id), and any it cannot fill."""
    name = fields["country"]
    values = {"{country}'s": name + ("'" if name.endswith("s") else "'s"), **{f"{{{k}}}": v for k, v in fields.items()}}
    for bracket, brace in BRACKETS.items():
        text = text.replace(bracket, brace)
    for placeholder, value in values.items():
        if placeholder in text:
            print(f"{where:<5} {placeholder:<15} ->  {value!r}")
            text = text.replace(placeholder, value)
    text = re.sub(r"(^|[.?!…]\s+)the ", lambda m: m.group(1) + "The ", text)
    for left in re.findall(r"\{[^}]*\}|\[[^\]]*\]", text):
        print(f"{where:<5} {left:<15} ->  UNFILLED (no value for it)")
    return text

def unlabelled(text):
    return not isinstance(text, str) or not re.search(r"[A-Za-z]", text)   # NaN or no letters ('02')

def scale_key(texts):
    return tuple(t for t in texts if not unlabelled(t))                      # labelled choices in code order

def fill_labels(codes, texts):
    fixes = LABEL_BANK.get(scale_key(texts), {})
    return [fixes.get(c, t) if unlabelled(t) else t for c, t in zip(codes, texts)]


COUNTRY_NOTE = re.compile(r"^([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)\s*:\s*")   # 'NL, US: Employers' -> offered only in NL and US

def offered_options(labels, country, verbose=False):
    """{code: text} of every valid (non-negative) option offered in `country`, in code order. An option marked for
    other countries only is dropped; an option marked for `country` keeps its text without the marker; a country note
    after the text ('Never; PL: 1-2 times a year') gives this country's wording if it names this country, else is cut.
    verbose=True prints every option it drops or changes."""
    out = {}
    for k, t in sorted(labels.items()):
        if k < 0:
            continue
        text = re.sub(r"^-?\d+\.\s+", "", t)
        m = COUNTRY_NOTE.match(text)
        if m:
            if country not in re.split(r"\s*,\s*", m[1]):
                if verbose:
                    print(f"  {int(k):>4}  {text!r}  ->  dropped: not offered in {country}")
                continue
            if verbose:
                print(f"  {int(k):>4}  {text!r}  ->  {text[m.end():]!r}")
            text = text[m.end():]
        own = persona_text(text, country)                              # country notes after the text
        if verbose and own != text:
            print(f"  {int(k):>4}  {text!r}  ->  {own!r}")
        out[int(k)] = own
    return out

def persona_text(label, country):
    """Persona value as this country's respondents saw it: '1. Male' -> 'Male'; 'Once a year; US: Once or twice a year'
    -> 'Once or twice a year' in the US, 'Once a year' elsewhere."""
    text = re.sub(r"^-?\d+\.\s+", "", str(label)).strip()
    m = re.fullmatch(r"(.*?)\s*\(([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)\s*:\s*(.*)\)", text)   # a note in brackets: '89 (US: 89 years or older)'
    if m:
        return m[3] if country in re.split(r"\s*,\s*", m[2]) else m[1]
    kept, ours = [], None
    
    for part in re.split(r";|(?<![A-Z]{2}),\s*(?=[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:)", text):
        part = part.strip()
        m = COUNTRY_NOTE.match(part)
        if not m:
            kept.append(part)                          # general wording
        elif country in re.split(r"\s*,\s*", m[1]):
            ours = part[m.end():]                      # this country's own wording
            if re.match(r"(incl|\+|also)", ours, re.I):  # a note that adds to the general wording: 'incl. nearly every week'
                ours = f"{kept[0]}, {ours.lstrip('+ ')}" if kept else ours
    return ours if ours is not None else "; ".join(kept)

def read_missing(path):
    """{variable: codes that are not answers} from GESIS's *_missing.txt: ' v1 (8, 9)', ' DE_v34 (996 THRU 999, 990)',
    ' v46 (LOWEST THRU -4, -1)'. Without the file ({}), only negative codes are non-answers."""
    missing = {}
    if path is None or not Path(path).exists():
        return missing
    for line in Path(path).read_text(encoding="latin-1").splitlines():
        match = re.match(r"\s*(\S+)\s*\((.*)\)", line.strip())
        if not match:
            continue
        codes = set()
        for part in match.group(2).upper().replace("LOWEST", "-999").replace("HIGHEST", "99999").split(","):
            if "THRU" in part:
                low, high = part.split("THRU")
                codes |= set(range(int(low), int(high) + 1))
            elif part.strip():
                codes.add(int(float(part)))
        missing[match.group(1)] = codes
    return missing

In [21]:
fn = 'ZA6770_v2-1-0'
data_dir = Path.cwd()/'data/issp2015'
data_fn = data_dir/f'raw/{fn}.dta'
bank_fn = data_dir/'question_bank.csv'
verified_fn = data_dir/'questions_verified.csv'
country = 'US'
long_fn = data_dir/f"issp2015_{country}_long.parquet"
missing = read_missing(data_dir/f'raw/{fn}_missing.txt')
missing["v1"], missing["ATTEND"], missing["VOTE_LE"]


({8, 9}, {0, 97, 98, 99}, {0, 7, 8, 9})

In [22]:
max_options = 15
countries = pd.read_csv(Path.cwd()/'data/issp_countries.csv', keep_default_na=False).set_index("c_alphan")
fields = {**countries.loc[country, ["country", "nationality", "parliament"]].to_dict(),   # this country's names
          "association": "the United States-Mexico-Canada Agreement (USMCA)"}             # 2023 Q38-Q43: data/issp2023/build_dataset.py

bank = pd.read_csv(bank_fn)

raw = pd.read_stata(data_fn, convert_categoricals=False)        # NEW: same file, answers as integer codes
labels = pd.io.stata.StataReader(data_fn).variable_labels()     # column -> GESIS label
with pd.io.stata.StataReader(data_fn) as reader:
    label_sets = reader.value_labels()                                   # label-set name -> {code: "N. text"}
    value_labels = {v: label_sets.get(s, {}) for v, s in zip(reader._varlist, reader._lbllist)}   # column -> {code: "N. text"}

repeated = [c for c, vl in value_labels.items() if len(set(vl.values())) < len(vl)]   # labels pandas cannot turn into categories (e.g. ISCO08)

wide = pd.read_stata(data_fn, columns=[c for c in raw.columns if c not in repeated])   # as stored; answers shown with GESIS's labels

fixes_fn = data_dir/'label_fixes.csv'                                  # labels the .dta cuts off, completed from the codebook
number = r"^-?\d+\.\s*"                                               # a label's own number prefix ('4. ')
if fixes_fn.exists():
    for fix in pd.read_csv(fixes_fn, keep_default_na=False).itertuples():
        vl = value_labels.get(fix.variable, {})
        old = vl.get(fix.code, "")
        if re.sub(number, "", old) != re.sub(number, "", fix.data_label):
            continue                                                   # label changed since the fix was made: leave it
        prefix = re.match(number, old)
        new = (prefix.group(0) if prefix else "") + re.sub(number, "", fix.fixed_label)
        value_labels[fix.variable] = {**vl, fix.code: new}
        if fix.variable in wide.columns and old in wide[fix.variable].cat.categories:
            wide[fix.variable] = wide[fix.variable].cat.rename_categories({old: new})   # the answers keep their label
        print(f"{fix.variable} {fix.code}: ...{old[-25:]!r} -> ...{new[-25:]!r}")

# GESIS's non-answer codes that are not negative (e.g. 8 Can't choose, 9 No answer, 0 Not available) -> negative
# (-1000 - code), so every rule below that treats negative codes as non-answers covers them too
missing = read_missing(next(data_dir.glob("raw/*_missing.txt"), None))
for c, codes in missing.items():
    codes = {k for k in codes if k >= 0}
    if c not in raw.columns or not codes or raw[c].dtype.kind not in "iuf":
        continue
    s = raw[c].astype("float64")
    raw[c] = s.where(~s.isin(codes), -1000 - s)
    value_labels[c] = {(-1000 - int(k) if k in codes else k): t for k, t in value_labels.get(c, {}).items()}

if "EDULEVEL" not in raw.columns:                                       # up to 2019 education is DEGREE: same role, its own categories
    wide, raw = wide.rename(columns={"DEGREE": "EDULEVEL"}), raw.rename(columns={"DEGREE": "EDULEVEL"})
    value_labels["EDULEVEL"], labels["EDULEVEL"] = value_labels["DEGREE"], labels["DEGREE"]

wide = wide[wide["c_alphan"] == country]
raw = raw.loc[wide.index]                                       # NEW: same rows as wide

wide["CASEID"] = wide["CASEID"].astype("int64")             # stored as a float; shown as 2.022008e+15 otherwise

persona = ["c_alphan", "SEX", "AGE", "EDULEVEL", "RELIGGRP", "ATTEND", "PARTY_LR", "VOTE_LE", "URBRURAL",
           "MARITAL", "MAINSTAT", "UNION", f"{country}_PRTY", f"{country}_INC", f"{country}_REG", f"{country}_ETHN1"]

questions = [c for c in wide.columns if re.fullmatch(r"[vV]\d+", c)]   # v1, v2, ... (2015 has one V16): the substantive questions

def all_options(v):
    """{code: text} of every option of question v in this country, in code order: the labelled options offered here
    (offered_options), plus every non-negative code without a label between the labelled ends or given by a respondent
    here (text NaN, filled by LABEL_BANK in the next cell). Codes labelled only for other countries stay out."""
    opts = offered_options(value_labels[v], country)
    given = set(raw.loc[raw[v] >= 0, v].astype(int))                   # codes respondents in this country gave
    labelled = {int(k) for k in value_labels[v] if k >= 0}              # codes with a value label (any country)
    scale = bank.loc[bank["v"] == v, "type"].eq("score").any()           # unlabelled points only exist on ordered scales
    span = set(range(min(opts), max(opts) + 1)) if opts and scale else set()   # every code between the labelled ends
    for c in (span - labelled) | (given - set(opts)):
        opts[c] = float("nan")
    return dict(sorted(opts.items()))

options = {v: all_options(v) for v in questions}                        # v -> {code: text}, every option in this country

wide[questions] = wide[questions].where(raw[questions] >= 0)                          # NEW: negative answers -> NaN (that cell only)

questions = [v for v in questions                                              # the one place questions are kept or dropped:
             if v in set(bank.loc[bank["type"] != "exclude", "v"])            #   in the bank and not marked exclude
             and len(options[v]) <= max_options]                              #   at most max_options offered options

bank = bank[bank["v"].isin(questions)]

# this country's names in place of placeholders, in question texts and option texts
bank["question"] = [fill_placeholders(t, fields, v) for v, t in zip(bank["v"], bank["question"])]
options = {v: {c: fill_placeholders(t, fields, v) if isinstance(t, str) else t for c, t in o.items()} for v, o in options.items()}

wide = wide[["CASEID"] + persona + questions]

wide[questions] = wide[questions].apply(lambda s: s.cat.set_categories(
    [value_labels[s.name].get(k, k) for k in options[s.name]]))         # categories = every option in this country, in code order (an unlabelled code is its own category)

# persona: number prefix removed, this country's wording taken from country notes, negative codes -> NaN
for c in persona[1:]:                                             # c_alphan stay as it is
    vl = value_labels.get(c, {})
    values = {}
    for k in sorted(raw[c].dropna().unique()):
        before = re.sub(r"^-?\d+\.\s+", "", str(vl.get(k, int(k)))).strip()
        if k < 0:
            values[k] = float("nan")
            print(f"{c:<10} {int(k):>6}  {before!r}  ->  NaN  ({int((raw[c] == k).sum())} respondents)")
        else:
            values[k] = persona_text(vl.get(k, int(k)), country)
            if values[k] != before:
                print(f"{c:<10} {int(k):>6}  {before!r}  ->  {values[k]!r}")
    wide[c] = raw[c].map(values)

long = wide.melt(id_vars=["CASEID"]+persona, value_vars=questions, var_name="v", value_name="answer")
long.insert(long.columns.get_loc("v") + 1, "label", long["v"].map(labels))
long["answer_code"] = raw[questions].melt()["value"].astype("Int64").to_numpy()   # CHANGED: code from the raw integers, not the label string
long = long.dropna(subset=["answer"])  # CHANGED: drop the NaN cells (negative or missing answers)

long.head(2)

v35 3: ...'ng times are decided at s' -> ...'ort notice by my employer'
EE_DEGR 5: ...'ucation, without secondar' -> ...'thout secondary education'
EE_DEGR 6: ...'ucation, with secondary e' -> ...' with secondary education'
EE_DEGR 11: ...'tutions of higher educati' -> ...'er education before 1992)'
HU_DEGR 3: ...'ithout high-school leavin' -> ...' high-school leaving exam'
HU_DEGR 4: ...'; vocational training fol' -> ...' high-school leaving exam'
HU_DEGR 6: ...'-school leaving exam, not' -> ...'econdary technical school'
HU_DEGR 7: ...'sional training; tertiary' -> ...'tertiary technical school'
IS_DEGR 4: ...' exams, the nationally co' -> ...'gagnfræðapróf, landspróf)'
IS_DEGR 5: ...'ary school level (e.g. ho' -> ...'e, shorter music studies)'
IS_DEGR 6: ...'am from the university br' -> ...'minary studies department'
IS_DEGR 8: ...'dary school level (e.g. n' -> ...'use training, trade exam)'
IS_DEGR 9: ...'of secondary school level' -> ...'ineer, medical secretary)'
IS_DEGR 12: ...

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,MAINSTAT,UNION,US_PRTY,US_INC,US_REG,US_ETHN1,v,label,answer,answer_code
0,2015008400000003,US,Male,72,"Lower level tertiary, first stage (also techni...",Catholic,Once a week,"Right, conservative",Yes,The suburbs or outskirts of a big city,...,Retired,"No, never",Republican (Romney),75.000-89.999 USD,New England,Italy,v1,Q1a Job is a way of earning money,Agree,2
1,2015008400000005,US,Female,55,"Upper level tertiary (Master, Doctor)",No religion,Never,"Left, center left",Yes,The suburbs or outskirts of a big city,...,In paid work,"No, never",Democrat (Obama),170.000 USD or more per year,New England,Italy,v1,Q1a Job is a way of earning money,Disagree,4


In [23]:
# bank: every option offered in this country, in code order; label fixes shown before -> after
info = bank.set_index("v")
rows = []
for v in questions:
    before = {int(k): re.sub(r"^-?\d+\.\s+", "", t) for k, t in sorted(value_labels[v].items()) if k >= 0}   # as the data has it
    before = dict(sorted((before | {c: float("nan") for c in options[v] if c not in before}).items()))      # + codes without a label
    opts = options[v]                                                                                       # country markers handled
    codes = list(opts)
    after = dict(zip(codes, fill_labels(codes, list(opts.values()))))                                       # unlabelled choices filled
    if any(unlabelled(t) for t in after.values()):                     # no LABEL_BANK entry: the data's numbered form for the whole scale
        after = {c: (t if re.match(rf"{c}\s", t) else f"{c}. {t}") if not unlabelled(t) else f"{c}." for c, t in after.items()}
    rows.append({"v": v,
                 "answer_code": codes,
                 "answer": [value_labels[v].get(k, str(k)) for k in codes],
                 "answer_text": list(after.values())})
    if before == after:
        continue
    print("=" * 100)
    print(f"{v} | {info.loc[v, 'type']} | {info.loc[v, 'data_label']}")
    if any(unlabelled(t) for t in after.values()):
        print("NO ENTRY IN LABEL_BANK FOR THIS SCALE")
    print("-" * 100)
    for c, b in before.items():
        print(f"{c:>4}  {str(b):<45} ->  {after.get(c, f'(dropped: not offered in {country})')}")
print("=" * 100)

bank = bank.merge(pd.DataFrame(rows), on="v")

# long: each person's answer text taken from bank, so the two always agree
text_of = {(r["v"], c): t for _, r in bank.iterrows() for c, t in zip(r["answer_code"], r["answer_text"])}
long["answer_text"] = [text_of[(v, c)] for v, c in zip(long["v"], long["answer_code"])]

# long: the question each answer belongs to, with its type, direction and every option offered
long = long.merge(
    bank[["v", "question", "type", "reversed", "answer_code", "answer_text"]]
        .rename(columns={"answer_code": "option_codes", "answer_text": "option_texts"}),
    on="v", how="left")
long.head(2)

v15 | choice | Q6 Reason for discrimination
----------------------------------------------------------------------------------------------------
   1  My age                                        ->  My age
   2  My race, ethnicity                            ->  My race, ethnicity
   3  My nationality                                ->  My nationality
   4  My sex                                        ->  My sex
   5  My religion                                   ->  My religion
   6  My disability/ mental or physical illness     ->  My disability/ mental or physical illness
   7  My family responsibilities                    ->  My family responsibilities
   8  My political beliefs                          ->  My political beliefs
   9  Other reason;TW: Other incl. education        ->  Other reason
v40 | score | Q20 How much use of past work experience and skills
----------------------------------------------------------------------------------------------------
   1  Almost none    

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,v,label,answer,answer_code,answer_text,question,type,reversed,option_codes,option_texts
0,2015008400000003,US,Male,72,"Lower level tertiary, first stage (also techni...",Catholic,Once a week,"Right, conservative",Yes,The suburbs or outskirts of a big city,...,v1,Q1a Job is a way of earning money,Agree,2,Agree,Please tick one box for each statement below t...,score,True,"[1, 2, 3, 4, 5]","[Strongly agree, Agree, Neither agree nor disa..."
1,2015008400000005,US,Female,55,"Upper level tertiary (Master, Doctor)",No religion,Never,"Left, center left",Yes,The suburbs or outskirts of a big city,...,v1,Q1a Job is a way of earning money,Disagree,4,Disagree,Please tick one box for each statement below t...,score,True,"[1, 2, 3, 4, 5]","[Strongly agree, Agree, Neither agree nor disa..."


In [24]:
long.shape

(72416, 27)

In [25]:
missing = bank[bank["answer_text"].apply(
    lambda texts: any(not isinstance(t, str) or not re.search(r"[A-Za-z]", t) for t in texts))]
missing[["v", "type", "data_label", "answer_code", "answer_text"]]


,v,type,data_label,answer_code,answer_text


### Check persona

In [26]:
bad_persona = {}
for c in persona:
    print(c)
    print(list(long[c].unique()))
    print("+" * 50)
    invalid = [
        v for v in long[c].dropna().unique()
        if re.search(r";|\b[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:|^-?\d+\.\s+|\b(No answer|Don't know|Refused|NAP|NAV|Not available|Can't choose)\b", str(v), re.I)
    ]
    if invalid:
        bad_persona[c] = invalid
if bad_persona:
    raise ValueError(f"Unnormalized persona values found: {bad_persona}")

c_alphan
['US']
++++++++++++++++++++++++++++++++++++++++++++++++++
SEX
['Male', 'Female']
++++++++++++++++++++++++++++++++++++++++++++++++++
AGE
['72', '55', '23', '71', '33', '60', '56', '31', '32', '37', '25', '76', '22', '75', '68', '46', '59', '58', '40', '36', '70', '28', '47', '20', '41', '26', '51', '39', '30', '78', '35', '44', '45', '50', '27', '66', '89', '29', '54', '48', '81', '69', '52', '21', '64', '62', '65', '43', '61', '57', '49', '67', '84', '38', '79', '77', '42', '19', '63', nan, '74', '34', '53', '82', '24', '80', '83', '87', '73', '86', '18', '85', '88']
++++++++++++++++++++++++++++++++++++++++++++++++++
EDULEVEL
['Lower level tertiary, first stage (also technical schools at a tertiary level)', 'Upper level tertiary (Master, Doctor)', 'Upper secondary (programs that allows entry to university)', 'Post secondary, non-tertiary (other upper secondary programs toward labour market or technical formation)', 'Lower secondary (secondary completed does not allow entry to 

ValueError: Unnormalized persona values found: {'US_INC': ['75.000-89.999 USD', '170.000 USD or more per year', '30.000-34.999 USD', '60.000-74.999 USD', '12.500-14.999 USD', '50.000-59.999 USD', '35.000-39.999 USD', '40.000-49.999 USD', '22.500-24.999 USD', '20.000-22.499 USD', '5.000-5.999 USD', '110.000-129.999 USD', '8.000-9.999 USD', '25.000-29.999 USD', '150.000-169.999 USD', '130.000-149.999 USD', '90.000-109.999 USD', '15.000-17.499 USD', '1.000-2.999 USD', '7.000-7.999 USD', '17.500-19.999 USD', '10.000-12499 USD', '3.000-3.999 USD', '6.000-6.999 USD', '4.000-4.999 USD']}

# Export verified

In [70]:
# ---- export for training: one row per answer, with the person's persona, the question and every option offered
columns = ["CASEID"] + persona + ["v", "label", "question", "type", 
                                  "reversed", "option_codes", "option_texts",
                                  "answer_code", "answer_text"
                                ]
long[columns].to_parquet(long_fn, index=False)
long_fn


PosixPath('/home/dw/github/decision-models-finetuning/data/issp2015/issp2015_US_long.parquet')

In [71]:
bank['options_old'] = bank['options']
bank[['v', 'data_label', 'question', 'options_old', 'answer_code', 'answer_text', 'type', 'reversed']].to_csv(verified_fn,index=False)
